# Assignment 1: Explainable AI for Protein Biology 

**Course:** Advanced Machine Learning for the Life Sciences (MBioAML)
**Dataset:** Extracellular Vesicle (EV) Protein Sorting
**Environment:** Google Colab (CPU runtime is sufficient)

---

## Overview

Machine learning models can predict complex biological outcomes — such as whether a protein is sorted into extracellular vesicles — with high accuracy. But in biology, prediction alone is not enough. We need to understand *why* a model makes a decision: which features drive the prediction, and does that align with known biology?

This assignment puts those ideas into practice. You will:

1. **Train** three models (Logistic Regression, Random Forest, MLP) on real protein biology data
2. **Visualize** global model behavior with PDP, ICE, ALE, and permutation importance
3. **Explain** individual predictions with **LIME** — and explore how its hyperparameters affect results
4. **Explain** with **SHAP** using model-agnostic methods (Sampling, Permutation, Kernel SHAP)
5. **Explain** with **model-specific SHAP** (TreeSHAP, DeepSHAP) and compare speed/reliability
6. **Reflect** on what each method reveals about the biology of EV protein sorting

---

## The Dataset: Extracellular Vesicle Protein Sorting

Extracellular vesicles (EVs) are small membrane-bound particles released by cells. They carry proteins, lipids, and nucleic acids and play key roles in cell-to-cell communication, immune signalling, and even cancer metastasis. Understanding *which proteins* get sorted into EVs — and what makes them EV-destined — is a major open question in cell biology.

**Task:** Binary classification — predict whether a protein will be sorted into EVs (`EV = 1`) or not (`EV = 0`) from protein biochemical and bioinformatic features.

**Features (93 total) include:**
- Physicochemical properties: length, molecular weight, hydrophobicity, isoelectric point, charge, GRAVY index
- Amino acid composition (20 amino acid frequencies)
- Secondary structure fractions (helix, sheet, turn)
- Surface accessibility: total/relative accessible surface area, exposed residue fractions
- Post-translational modifications (PTMs): phosphorylation, ubiquitination, glycosylation, etc.
- Protein domain annotations: coiled-coil, transmembrane, EGF, RRM, etc.
- Solubility and aggregation propensity scores

---

## How to read this notebook

- **Explanatory cells** (like this one) introduce each concept and explain *why* we do things.
- **Code cells** contain the implementation — read them carefully before running.
- Cells marked with ✏️ **STUDENT TASK** require you to write code or answer questions.
- Questions marked **Q** should be answered directly in the notebook (add a markdown cell or use the provided answer box).

> ⚠️ **Colab note:** Run cells top-to-bottom. If your runtime disconnects, restart from Section 0.

---

## Table of Contents

| Section | Topic |
|---------|-------|
| [0. Setup](#setup) | Install packages, set seeds |
| [1. Data](#data) | Load, explore, and understand the EV dataset |
| [2. Models](#models) | Train Logistic Regression, Random Forest, MLP |
| [⬛ Day 1 Assignment](#day1) | Open exploration spot — complete before Day 2 |
| [3. Global Explanations](#global) | PDP, ICE, ALE, Permutation Importance |
| [4. LIME](#lime) | Local explanations, perturbations, kernel, fidelity, complexity |
| [5. SHAP Agnostic](#shap-ag) | Sampling, Permutation, Kernel SHAP; background sample effects |
| [6. SHAP Specific](#shap-sp) | TreeSHAP, DeepSHAP; speed & reliability vs Kernel SHAP |
| [7. Synthesis](#synthesis) | Compare methods, biological interpretation |


---
## Section 0: Setup & Installation <a id='setup'></a>

We install all required packages in one go. Copy-paste and run this cell first on Google Colab. It may take 1–2 minutes.

> **Why so many packages?** LIME and SHAP each come with their own dependencies. `torch` is needed for the MLP. `alibi` and `PyALE` provide ALE plots. `interpret` gives us PDP and ICE. `matplotlib` and `seaborn` for all visualization.


In [1]:
# ── 0.1  Install all required packages ───────────────────────────────────────
# Run this cell first. All output is suppressed for readability.
# If a package fails to install, remove the '> /dev/null 2>&1' and re-run to see the error.

!pip install -q numpy pandas "scikit-learn>=1.3" matplotlib seaborn > /dev/null 2>&1
!pip install -q torch > /dev/null 2>&1
!pip install -q lime > /dev/null 2>&1
!pip install -q shap > /dev/null 2>&1
!pip install -q PyALE > /dev/null 2>&1
!pip install -q tqdm joblib > /dev/null 2>&1

print("✅ All packages installed.")

✅ All packages installed.


In [2]:
# ── 0.2  Imports and global settings ─────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import warnings, time, random, copy
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")

# Sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    classification_report, RocCurveDisplay, confusion_matrix
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import train_test_split

# LIME
from lime.lime_tabular import LimeTabularExplainer

# SHAP
import shap

print("✅ All imports successful.")


PyTorch device: cpu
✅ All imports successful.


---
## Section 1: Data Loading and Exploration <a id='data'></a>

### 1.1 Loading the data

We train on a large dataset of annotated proteins and evaluate on a held-out test set from the Dhondt et al. (2021) study — a real external validation set. This is important: the test proteins come from a *different study* than the training proteins, so the model must generalise.

> **Upload instructions (Google Colab):**
> 1. Click the 📁 folder icon on the left sidebar
> 2. Upload `training_remaining.csv` and `test_Dhondt2021.csv`
> 3. Then run the cell below

Alternatively, if you store them in Google Drive:
```python
from google.colab import drive
drive.mount('/content/drive')
# Then set DATA_DIR to your drive path
```


In [ ]:
# ── 1.1  Load train and test data ────────────────────────────────────────────
DATA_DIR = Path("/content")   # default Colab upload location — change if needed

train_df = pd.read_csv(DATA_DIR / "training_remaining.csv")
test_df  = pd.read_csv(DATA_DIR / "test_Dhondt2021.csv")

TARGET  = "EV"
ID_COL  = "id"
FEATURE_COLS = [c for c in train_df.columns if c not in [TARGET, ID_COL]]

X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = train_df[TARGET].values
X_test  = test_df[FEATURE_COLS].values.astype(np.float32)
y_test  = test_df[TARGET].values

print(f"Training set : {X_train.shape[0]:,} samples  |  {X_train.shape[1]} features")
print(f"Test set     : {X_test.shape[0]:,} samples")
print(f"Feature list (first 10): {FEATURE_COLS[:10]}")


### 1.2 Class balance

Class imbalance is common in biology datasets. An imbalanced dataset means a model can appear to perform well simply by predicting the majority class. We must check the label distribution before trusting any accuracy number.


In [ ]:
# ── 1.2  Class balance ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (split_name, y) in zip(axes, [("Train", y_train), ("Test (Dhondt2021)", y_test)]):
    counts = np.bincount(y)
    bars = ax.bar(["Non-EV (0)", "EV (1)"], counts, color=["#4C72B0", "#DD8452"], edgecolor="black")
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                f"{cnt:,}\n({100*cnt/len(y):.1f}%)", ha="center", va="bottom", fontsize=10)
    ax.set_title(f"Label distribution — {split_name}", fontsize=12)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.savefig("class_balance.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test  positive rate: {y_test.mean():.3f}")


**Q1.** Is the dataset balanced? How will class imbalance affect model training and evaluation? Which metric is most reliable when classes are imbalanced — accuracy, F1, or AUC? Explain your reasoning.

> *Your answer here*


### 1.3 Feature overview

Let's look at some biologically interesting features to build intuition about what the model might learn.


In [ ]:
# ── 1.3  Feature distributions for selected biological features ──────────────
BIO_FEATURES = [
    "length", "molecular_weight", "Gravy", "Isoelectric_point",
    "Probability_solubility", "Aggregation_propensity",
    "thsa_netsurfp2", "disorder",
    "transmembrane", "Phosphorylation_all"
]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for ax, feat in zip(axes, BIO_FEATURES):
    for label, color, name in [(0, "#4C72B0", "Non-EV"), (1, "#DD8452", "EV")]:
        mask = y_train == label
        vals = train_df.loc[mask, feat]
        ax.hist(vals, bins=30, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel("Value", fontsize=8)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=8)
plt.suptitle("Feature distributions: EV vs Non-EV proteins", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("feature_distributions.png", dpi=120, bbox_inches="tight")
plt.show()


**Q2.** Looking at the feature distributions, which features seem most different between EV and non-EV proteins? Can you explain why those features might biologically distinguish EV-sorted proteins? (Hint: think about what extracellular vesicles are and how they form.)

> *Your answer here*


In [ ]:
# ── 1.4  Correlation heatmap (top 20 features by variance) ──────────────────
top_feat_idx = np.argsort(X_train.var(axis=0))[::-1][:20]
top_feat_names = [FEATURE_COLS[i] for i in top_feat_idx]
corr = pd.DataFrame(X_train[:, top_feat_idx], columns=top_feat_names).corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0,
            annot=False, linewidths=0.3, ax=ax, vmin=-1, vmax=1)
ax.set_title("Pairwise feature correlations (top 20 by variance)", fontsize=13)
plt.tight_layout()
plt.savefig("feature_correlations.png", dpi=120, bbox_inches="tight")
plt.show()

print("Highest absolute correlations:")
corr_long = corr.abs().unstack().sort_values(ascending=False)
corr_long = corr_long[corr_long < 1.0].drop_duplicates()
print(corr_long.head(10).to_string())


**Q3.** Many features are strongly correlated. Name two pairs. Why does feature correlation matter for LIME and SHAP? (We will return to this question in Sections 4–6.)

> *Your answer here*


---
## Section 2: Model Training <a id='models'></a>

We train three models of increasing complexity:

| Model | Why we use it |
|-------|--------------|
| **Logistic Regression** | Linear baseline; coefficients are directly interpretable |
| **Random Forest** | Non-linear ensemble; natively compatible with TreeSHAP |
| **MLP (PyTorch)** | Deep network; compatible with DeepSHAP and LIME |

All models are trained on the same features and evaluated on the same test set. We use `class_weight='balanced'` for sklearn models and a weighted loss for the MLP to account for class imbalance.

### 2.1 Logistic Regression


In [ ]:
# ── 2.1  Logistic Regression ──────────────────────────────────────────────────
# We wrap with a StandardScaler — LR is sensitive to feature scale.
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=0.1,          # regularization
        random_state=SEED
    ))
])

lr_pipeline.fit(X_train, y_train)
lr_probs = lr_pipeline.predict_proba(X_test)[:, 1]
lr_preds = (lr_probs >= 0.5).astype(int)

print("Logistic Regression — Test set performance")
print(f"  AUC  : {roc_auc_score(y_test, lr_probs):.4f}")
print(f"  F1   : {f1_score(y_test, lr_preds):.4f}")
print(f"  Acc  : {accuracy_score(y_test, lr_preds):.4f}")
print(classification_report(y_test, lr_preds, target_names=["Non-EV","EV"]))


### 2.2 Random Forest


In [ ]:
# ── 2.2  Random Forest ───────────────────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=SEED
)

rf_model.fit(X_train, y_train)
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = (rf_probs >= 0.5).astype(int)

print("Random Forest — Test set performance")
print(f"  AUC  : {roc_auc_score(y_test, rf_probs):.4f}")
print(f"  F1   : {f1_score(y_test, rf_preds):.4f}")
print(f"  Acc  : {accuracy_score(y_test, rf_preds):.4f}")
print(classification_report(y_test, rf_preds, target_names=["Non-EV","EV"]))


### 2.3 MLP (PyTorch)

We build a small fully-connected network. The architecture is intentionally simple — three hidden layers with ReLU activations and dropout for regularization. We also build a **scikit-learn compatible wrapper** so that LIME (which expects a `predict_proba` function) can use this model without any modifications.


In [ ]:
# ── 2.3a  MLP architecture ────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class EVNet(nn.Module):
    """
    Simple 3-layer MLP for EV protein classification.
    Output: single logit (apply sigmoid for probability).
    """
    def __init__(self, input_dim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)   # shape: (B,)


INPUT_DIM = X_train.shape[1]
mlp = EVNet(INPUT_DIM).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in mlp.parameters() if p.requires_grad)
print(f"MLP: {total_params:,} trainable parameters")
print(mlp)


In [ ]:
# ── 2.3b  Preprocess: scale inputs ────────────────────────────────────────────
# The MLP shares the same scaler as Logistic Regression.
mlp_scaler = StandardScaler()
X_train_scaled = mlp_scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled  = mlp_scaler.transform(X_test).astype(np.float32)

# Class weights for imbalanced training
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()],
                           dtype=torch.float32).to(DEVICE)

# Data loaders
train_tensor = TensorDataset(
    torch.tensor(X_train_scaled), torch.tensor(y_train, dtype=torch.float32))
test_tensor  = TensorDataset(
    torch.tensor(X_test_scaled),  torch.tensor(y_test,  dtype=torch.float32))

train_loader = DataLoader(train_tensor, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_tensor,  batch_size=512, shuffle=False)


In [ ]:
# ── 2.3c  Training loop ───────────────────────────────────────────────────────
def train_mlp(model, train_loader, n_epochs=30, lr=1e-3, pos_weight=None):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    best_loss = float("inf")
    best_state = None

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        epoch_loss /= len(train_loader)

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = copy.deepcopy(model.state_dict())

        if epoch % 5 == 0:
            print(f"  Epoch {epoch:3d}/{n_epochs}  loss={epoch_loss:.4f}")

    model.load_state_dict(best_state)
    return model


print("Training MLP…")
mlp = train_mlp(mlp, train_loader, n_epochs=30, pos_weight=pos_weight)
print("Done.")


In [ ]:
# ── 2.3d  MLP evaluation + sklearn wrapper ────────────────────────────────────
def mlp_predict_proba(X: np.ndarray) -> np.ndarray:
    """
    sklearn-compatible predict_proba for the MLP.
    Accepts a raw (unscaled) numpy array, scales internally.
    Returns shape (N, 2): [P(non-EV), P(EV)].
    """
    X_sc = mlp_scaler.transform(X).astype(np.float32)
    with torch.no_grad():
        logits = mlp(torch.tensor(X_sc).to(DEVICE))
        probs  = torch.sigmoid(logits).cpu().numpy()
    return np.column_stack([1 - probs, probs])


mlp_probs = mlp_predict_proba(X_test)[:, 1]
mlp_preds = (mlp_probs >= 0.5).astype(int)

print("MLP — Test set performance")
print(f"  AUC  : {roc_auc_score(y_test, mlp_probs):.4f}")
print(f"  F1   : {f1_score(y_test, mlp_preds):.4f}")
print(f"  Acc  : {accuracy_score(y_test, mlp_preds):.4f}")
print(classification_report(y_test, mlp_preds, target_names=["Non-EV","EV"]))


### 2.4 Model comparison


In [ ]:
# ── 2.4  Compare all three models ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curves
for name, probs, color in [
    ("Logistic Regression", lr_probs, "#4C72B0"),
    ("Random Forest",       rf_probs, "#55A868"),
    ("MLP",                 mlp_probs,"#DD8452"),
]:
    RocCurveDisplay.from_predictions(y_test, probs, name=name,
                                      ax=axes[0], color=color)
axes[0].plot([0,1],[0,1],"k--",lw=1)
axes[0].set_title("ROC curves — Test set")

# Bar chart: AUC / F1
models  = ["LR", "RF", "MLP"]
aucs = [roc_auc_score(y_test, p) for p in [lr_probs, rf_probs, mlp_probs]]
f1s  = [f1_score(y_test, p > .5)  for p in [lr_probs, rf_probs, mlp_probs]]

x = np.arange(len(models))
w = 0.35
axes[1].bar(x - w/2, aucs, w, label="AUC",  color="#4C72B0", edgecolor="black")
axes[1].bar(x + w/2, f1s,  w, label="F1",   color="#DD8452", edgecolor="black")
axes[1].set_xticks(x); axes[1].set_xticklabels(models, fontsize=12)
axes[1].set_ylim(0, 1); axes[1].set_ylabel("Score")
axes[1].set_title("AUC and F1 — Test set")
axes[1].legend(); axes[1].axhline(0.5, color="gray", lw=1, ls="--")

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


---
## ⬛ Day 1 Assignment <a id='day1'></a>


---
## Section 3: Global Explanations — PDP, ICE, ALE, Permutation Importance <a id='global'></a>

In the lectures, we introduced four global explanation methods. Here we apply all four to the EV protein dataset. Each method answers a slightly different question:

| Method | Question answered |
|--------|------------------|
| **Permutation Importance** | Which features, when scrambled, hurt the model most? |
| **PDP** | On average across all proteins, how does feature X affect predicted EV probability? |
| **ICE** | For *individual* proteins, how does the prediction change as we vary feature X? |
| **ALE** | Like PDP, but restricted to realistic feature regions — safer for correlated features |

We use the **Random Forest** as our reference model for this section because it is the best-performing non-linear model and natively supports TreeSHAP later.


### 3.1 Permutation Feature Importance

Permutation importance is model-agnostic and honest: it directly measures how much the model's test performance drops when a feature's information is destroyed. This avoids the impurity-bias in tree built-in importances.


In [ ]:
# ── 3.1  Permutation feature importance ──────────────────────────────────────
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    rf_model, X_test, y_test,
    n_repeats=10,
    scoring="roc_auc",
    random_state=SEED,
    n_jobs=-1
)

# Sort by mean importance
perm_mean = perm_result.importances_mean
perm_std  = perm_result.importances_std
order     = np.argsort(perm_mean)[::-1][:20]

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(
    [FEATURE_COLS[i] for i in order[::-1]],
    perm_mean[order[::-1]],
    xerr=perm_std[order[::-1]],
    color="#4C72B0", edgecolor="black", capsize=4
)
ax.set_xlabel("Mean decrease in AUC when feature is shuffled")
ax.set_title("Permutation Feature Importance — Random Forest (top 20)")
ax.axvline(0, color="gray", lw=1, ls="--")
plt.tight_layout()
plt.savefig("permutation_importance.png", dpi=120, bbox_inches="tight")
plt.show()

print("Top 10 features by permutation importance:")
for rank, i in enumerate(order[:10], 1):
    print(f"  {rank:2d}. {FEATURE_COLS[i]:<35s}  {perm_mean[i]:.4f} ± {perm_std[i]:.4f}")


**Q4.** Which features are most important according to permutation importance? Does this list match your hypotheses from DAY1-Q3? Interpret 2–3 of the top features biologically.

> *Your answer here*


### 3.2 Partial Dependence Plots (PDP) and ICE

PDPs show the *average* predicted EV probability across the full training set as one feature varies. ICE plots show the same but for *individual proteins*, revealing heterogeneity that the average hides.

Recall the key limitation: PDP averages over the entire data, including potentially unrealistic feature combinations (especially when features are correlated).


In [ ]:
# ── 3.2  PDP + ICE for top 4 features ────────────────────────────────────────
TOP4 = [FEATURE_COLS[i] for i in order[:4]]
TOP4_IDX = [order[i] for i in range(4)]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for col, feat_idx in enumerate(TOP4_IDX):
    feat_name = FEATURE_COLS[feat_idx]

    # PDP
    PartialDependenceDisplay.from_estimator(
        rf_model, X_train, [feat_idx],
        kind="average",
        ax=axes[0, col],
        line_kw={"color": "#4C72B0", "lw": 2.5}
    )
    axes[0, col].set_title(f"PDP — {feat_name}", fontsize=10)
    axes[0, col].set_xlabel(feat_name, fontsize=9)

    # ICE (subsample 200 proteins for visibility)
    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(len(X_train), size=200, replace=False)
    PartialDependenceDisplay.from_estimator(
        rf_model, X_train[sample_idx], [feat_idx],
        kind="individual",
        ax=axes[1, col],
        line_kw={"color": "#DD8452", "alpha": 0.15, "lw": 1}
    )
    axes[1, col].set_title(f"ICE (200 samples) — {feat_name}", fontsize=10)
    axes[1, col].set_xlabel(feat_name, fontsize=9)

axes[0, 0].set_ylabel("Predicted EV probability (average)")
axes[1, 0].set_ylabel("Predicted EV probability (per protein)")

plt.suptitle("PDP (top row) and ICE (bottom row) for top 4 features", fontsize=13)
plt.tight_layout()
plt.savefig("pdp_ice.png", dpi=120, bbox_inches="tight")
plt.show()


**Q5.** Compare the PDP and ICE plots for each feature.
- Where do the ICE curves *spread out* compared to the PDP? What does that spread tell you?
- For which feature(s) does the PDP look like a flat line but the ICE shows strong individual effects? This is a sign of a **subgroup** structure — some proteins respond strongly while others don't.
- What does a **crossing of ICE curves** mean? (Hint: what happens to the PDP shape when many curves cross?)

> *Your answer here*


### 3.3 Accumulated Local Effects (ALE)

ALE avoids the PDP problem of unrealistic feature combinations. Instead of varying a feature for *every* protein, it only looks at small intervals and averages the local effect *within* that interval — so it stays close to the real data distribution. For biology, where features are heavily correlated (e.g. many amino acid frequencies sum to 1), ALE is often more trustworthy than PDP.


In [ ]:
# ── 3.3  ALE plots ────────────────────────────────────────────────────────────
from PyALE import ale

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, feat_idx in zip(axes, TOP4_IDX):
    feat_name = FEATURE_COLS[feat_idx]
    df_train  = pd.DataFrame(X_train, columns=FEATURE_COLS)

    ale_result = ale(
        X=df_train,
        model=rf_model,
        feature=[feat_name],
        feature_type="continuous",
        include_CI=True,
        C=0.95,
        plot=False
    )
    ale_x   = ale_result.index.values
    ale_eff  = ale_result["eff"].values
    ale_low  = ale_result["lowerCI_95%"].values if "lowerCI_95%" in ale_result.columns else ale_eff
    ale_high = ale_result["upperCI_95%"].values if "upperCI_95%" in ale_result.columns else ale_eff

    ax.plot(ale_x, ale_eff, color="#4C72B0", lw=2, label="ALE")
    ax.fill_between(ale_x, ale_low, ale_high, alpha=0.2, color="#4C72B0")
    ax.axhline(0, color="gray", ls="--", lw=1)
    ax.set_title(f"ALE — {feat_name}", fontsize=10)
    ax.set_xlabel(feat_name, fontsize=9)

axes[0].set_ylabel("ALE effect on predicted EV probability")
plt.suptitle("Accumulated Local Effects (ALE) — top 4 features", fontsize=13)
plt.tight_layout()
plt.savefig("ale_plots.png", dpi=120, bbox_inches="tight")
plt.show()


**Q6.** Compare the PDP and ALE plots for the same features. Do they tell the same story?

If they differ, consider: which features are most correlated with the feature being plotted? (Check your correlation heatmap from Section 1.4.) When features are strongly correlated, PDP can be misleading — it evaluates the model on inputs that never occur in real proteins.

> *Your answer here*


---
## Section 4: LIME — Local Interpretable Model-Agnostic Explanations <a id='lime'></a>

LIME explains **one prediction at a time**. For a protein you want to understand, LIME:
1. Creates many slightly perturbed copies of that protein (changing feature values)
2. Asks the model for predictions on all those perturbed copies
3. Weights perturbed copies by their distance to the original protein (closer = higher weight)
4. Fits a simple **sparse linear model** on the weighted perturbed dataset
5. The coefficients of that linear model are the feature attributions

LIME is **model-agnostic** — it treats any model as a black box and only calls `predict_proba`.

We will use the **Random Forest** as our primary model in this section (also try the MLP if you want).

---

### 4.1 Setting up the LIME explainer

The `LimeTabularExplainer` needs:
- Training data (to understand feature distributions and ranges)
- Feature names and mode (`classification` or `regression`)
- Class names (for display)


In [ ]:
# ── 4.1  Create the LIME explainer ───────────────────────────────────────────
lime_explainer = LimeTabularExplainer(
    training_data   = X_train,
    feature_names   = FEATURE_COLS,
    class_names     = ["Non-EV", "EV"],
    mode            = "classification",
    discretize_continuous = True,
    random_state    = SEED
)

print("LIME explainer created.")
print(f"Number of training samples used: {X_train.shape[0]:,}")
print(f"Number of features: {len(FEATURE_COLS)}")


### 4.2 Explaining a single prediction

Let's pick one EV protein that the Random Forest predicts correctly and one that it gets wrong. Comparing the two is informative: the model may be correct for the right reasons on a true positive, but may rely on spurious features for a false negative.


In [ ]:
# ── 4.2  Select example proteins ─────────────────────────────────────────────
# Find a confidently correct EV prediction (true positive)
ev_mask   = y_test == 1
tp_mask   = (rf_probs >= 0.7) & ev_mask
tp_indices = np.where(tp_mask)[0]
tp_idx     = tp_indices[0] if len(tp_indices) > 0 else np.where(ev_mask)[0][0]

# Find a false negative (EV protein the model misses)
fn_mask   = (rf_probs < 0.4) & ev_mask
fn_indices = np.where(fn_mask)[0]
fn_idx     = fn_indices[0] if len(fn_indices) > 0 else np.where(ev_mask)[0][-1]

print(f"True positive example — test index {tp_idx}")
print(f"  True label : {y_test[tp_idx]}  |  RF probability : {rf_probs[tp_idx]:.3f}")
print()
print(f"False negative example — test index {fn_idx}")
print(f"  True label : {y_test[fn_idx]}  |  RF probability : {rf_probs[fn_idx]:.3f}")


In [ ]:
# ── 4.2b  LIME explanation for the true positive ─────────────────────────────
exp_tp = lime_explainer.explain_instance(
    data_row        = X_test[tp_idx],
    predict_fn      = rf_model.predict_proba,
    num_features    = 15,       # number of features in the explanation
    num_samples     = 1000,     # number of perturbations
    distance_metric = "euclidean"
)

fig = exp_tp.as_pyplot_figure(label=1)
fig.set_size_inches(10, 5)
plt.title(f"LIME — True Positive (protein {tp_idx}, RF prob = {rf_probs[tp_idx]:.2f})",
          fontsize=12)
plt.tight_layout()
plt.savefig("lime_tp.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nTop contributing features (True Positive):")
for feat, weight in exp_tp.as_list(label=1):
    print(f"  {feat:<45s}  {weight:+.4f}")


In [ ]:
# ── 4.2c  LIME explanation for the false negative ────────────────────────────
exp_fn = lime_explainer.explain_instance(
    data_row        = X_test[fn_idx],
    predict_fn      = rf_model.predict_proba,
    num_features    = 15,
    num_samples     = 1000,
    distance_metric = "euclidean"
)

fig = exp_fn.as_pyplot_figure(label=1)
fig.set_size_inches(10, 5)
plt.title(f"LIME — False Negative (protein {fn_idx}, RF prob = {rf_probs[fn_idx]:.2f})",
          fontsize=12)
plt.tight_layout()
plt.savefig("lime_fn.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nTop contributing features (False Negative):")
for feat, weight in exp_fn.as_list(label=1):
    print(f"  {feat:<45s}  {weight:+.4f}")


**Q7.** Compare the LIME explanations for the true positive and false negative:
- Which features push toward EV in the true positive case?
- For the false negative, do the features that push *against* EV make biological sense?
- Can you generate a hypothesis about why the model missed this protein?

> *Your answer here*


### 4.3 Effect of number of perturbations (`num_samples`)

The number of perturbations controls how well LIME estimates the local decision boundary. Too few samples → noisy linear fit → unreliable attributions. Too many → slow but more stable.

**Key question:** At what sample size does the LIME explanation stabilise?


In [ ]:
# ── 4.3  Effect of num_samples on LIME explanation ───────────────────────────
sample_sizes = [50, 100, 250, 500, 1000, 2000, 5000]
TOP_FEAT_NAME = FEATURE_COLS[order[0]]   # most important feature from permutation importance

lime_weights_by_n = {}

for n in sample_sizes:
    exp = lime_explainer.explain_instance(
        data_row     = X_test[tp_idx],
        predict_fn   = rf_model.predict_proba,
        num_features = len(FEATURE_COLS),
        num_samples  = n,
    )
    weights_dict = dict(exp.as_list(label=1))
    lime_weights_by_n[n] = weights_dict

# Track the weight of 3 features across sample sizes
TRACK_FEATS_PARTIAL = [f for f in lime_weights_by_n[5000].keys()
                        if any(tf in f for tf in [FEATURE_COLS[order[i]] for i in range(3)])][:3]
if not TRACK_FEATS_PARTIAL:
    TRACK_FEATS_PARTIAL = list(lime_weights_by_n[5000].keys())[:3]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#4C72B0", "#DD8452", "#55A868"]
for feat, color in zip(TRACK_FEATS_PARTIAL, colors):
    vals = [lime_weights_by_n[n].get(feat, np.nan) for n in sample_sizes]
    ax.plot(sample_sizes, vals, "o-", color=color, lw=2, label=feat[:40])

ax.set_xscale("log")
ax.set_xlabel("Number of perturbations (num_samples)", fontsize=11)
ax.set_ylabel("LIME feature weight", fontsize=11)
ax.set_title("LIME stability: feature weight vs number of perturbations", fontsize=12)
ax.legend(fontsize=8, loc="upper right")
ax.axhline(0, color="gray", lw=1, ls="--")
plt.tight_layout()
plt.savefig("lime_num_samples.png", dpi=120, bbox_inches="tight")
plt.show()


**Q8.** At what number of perturbations do the LIME weights stabilise? What happens when you use very few samples (e.g. 50)? What is the practical implication for running LIME in a study?

> *Your answer here*


### 4.4 Effect of kernel width (distance / neighborhood size)

LIME uses a **kernel function** to weight perturbations by their distance to the original sample. Closer perturbations get higher weight. The **kernel width** controls the *size* of the local neighborhood:

- **Small kernel width** → only very nearby perturbations count → very local, but noisy
- **Large kernel width** → distant perturbations also count → smoother, but loses locality

In `lime`, the kernel width can be set via the `kernel_width` parameter of `LimeTabularExplainer`. The default is `np.sqrt(n_features) * 0.75`.


In [ ]:
# ── 4.4  Effect of kernel_width on LIME explanation ──────────────────────────
kernel_widths = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 25.0]
lime_weights_by_kw = {}

for kw in kernel_widths:
    exp_kw = LimeTabularExplainer(
        training_data         = X_train,
        feature_names         = FEATURE_COLS,
        class_names           = ["Non-EV", "EV"],
        mode                  = "classification",
        kernel_width          = kw,
        discretize_continuous = True,
        random_state          = SEED
    ).explain_instance(
        data_row     = X_test[tp_idx],
        predict_fn   = rf_model.predict_proba,
        num_features = len(FEATURE_COLS),
        num_samples  = 1000,
    )
    lime_weights_by_kw[kw] = dict(exp_kw.as_list(label=1))

# Track same 3 features
fig, ax = plt.subplots(figsize=(10, 5))
for feat, color in zip(TRACK_FEATS_PARTIAL, colors):
    vals = [lime_weights_by_kw[kw].get(feat, np.nan) for kw in kernel_widths]
    ax.plot(kernel_widths, vals, "o-", color=color, lw=2, label=feat[:40])

ax.set_xscale("log")
ax.set_xlabel("Kernel width", fontsize=11)
ax.set_ylabel("LIME feature weight", fontsize=11)
ax.set_title("LIME stability: feature weight vs kernel width", fontsize=12)
ax.legend(fontsize=8)
ax.axhline(0, color="gray", lw=1, ls="--")
plt.tight_layout()
plt.savefig("lime_kernel_width.png", dpi=120, bbox_inches="tight")
plt.show()

default_kw = np.sqrt(len(FEATURE_COLS)) * 0.75
print(f"Default kernel width: {default_kw:.2f}")


**Q9.** How do the feature attributions change as you increase the kernel width?
- At very small kernel width, why might the explanation be noisy?
- At very large kernel width, the local linear model effectively approximates the *global* behavior — is that still useful?
- Relate this to the LIME limitation discussed in the lecture: "There is no principled rule for choosing this width."

> *Your answer here*


### 4.5 Complexity: number of features in the explanation

LIME enforces **sparsity** via the `num_features` parameter — it only includes the `num_features` most important features in the linear surrogate. This is the "Ω(g)" complexity penalty from the lecture.

More features → a potentially more accurate local surrogate, but a harder-to-interpret explanation. Fewer features → simpler but possibly incomplete explanation.


In [ ]:
# ── 4.5  Effect of num_features (complexity) ─────────────────────────────────
# ========================== STUDENT TASK START ==========================
# STUDENT TASK 4.5: Run LIME for num_features = [3, 5, 10, 20, 30, 50]
# For each, compute the explanation and print the top features and weights.
# Then: compute the "fidelity" for each — defined below in section 4.6.
# Plot how the fidelity score changes as num_features increases.
# What do you observe? Is there a sweet spot?

num_features_list = [3, 5, 10, 20, 30, 50]

for nf in num_features_list:
    exp_nf = lime_explainer.explain_instance(
        data_row     = X_test[tp_idx],
        predict_fn   = rf_model.predict_proba,
        num_features = nf,
        num_samples  = 1000,
    )
    # The explanation has a local_pred attribute — the surrogate's prediction
    surrogate_pred = exp_nf.local_pred[1] if hasattr(exp_nf, "local_pred") else None
    true_pred      = rf_model.predict_proba(X_test[tp_idx:tp_idx+1])[0, 1]
    fidelity_err   = abs(surrogate_pred - true_pred) if surrogate_pred is not None else None

    feat_list = exp_nf.as_list(label=1)
    print(f"num_features={nf:3d}  |  RF prob={true_pred:.3f}  "
          f"|  Surrogate pred={surrogate_pred:.3f if surrogate_pred else 'N/A':.3f}  "
          f"|  |error|={fidelity_err:.4f if fidelity_err else 'N/A':.4f}")
# ========================== STUDENT TASK END ============================


### 4.6 Fidelity: how well does the surrogate match the black box?

**Fidelity** measures how accurately the local linear surrogate reproduces the black-box model's prediction *near the sample being explained*. High fidelity means the surrogate is a faithful local approximation. Low fidelity means the surrogate is misleading — you are looking at the wrong model.

LIME exposes its local predictions via `exp.local_pred`. We can also evaluate fidelity more rigorously by asking: across the *perturbed neighborhood*, does the surrogate agree with the black box?


In [ ]:
# ── 4.6  Fidelity measurement ─────────────────────────────────────────────────
def compute_lime_fidelity(explainer, instance, predict_fn,
                           num_features=15, num_samples=1000,
                           n_eval_samples=200, seed=SEED):
    """
    Compute LIME fidelity: correlation between black-box and surrogate predictions
    on a set of perturbed samples drawn around `instance`.

    Returns
    -------
    correlation : Pearson r between surrogate and black-box predictions
    mae         : mean absolute error between surrogate and black-box predictions
    """
    exp = explainer.explain_instance(
        data_row     = instance,
        predict_fn   = predict_fn,
        num_features = num_features,
        num_samples  = num_samples,
    )

    # Draw fresh perturbations — mimic what LIME does internally
    rng   = np.random.default_rng(seed)
    scale = np.std(explainer.training_data, axis=0)
    scale[scale == 0] = 1.0
    noise = rng.normal(0, 1, (n_eval_samples, instance.shape[0]))
    perturbed = instance + noise * scale * 0.5  # local neighborhood

    # Black-box predictions on perturbed samples
    bb_preds     = predict_fn(perturbed)[:, 1]

    # Surrogate predictions: linear model on same perturbed samples
    feat_weights = dict(exp.as_list(label=1))
    intercept    = exp.intercept[1] if hasattr(exp, "intercept") else 0.0

    # Build a dense weight vector
    w_vec = np.zeros(instance.shape[0])
    for feat_str, weight in feat_weights.items():
        # LIME discretises features — match on feature name in the string
        for fi, fname in enumerate(FEATURE_COLS):
            if fname in feat_str:
                w_vec[fi] = weight
                break

    surrogate_preds = perturbed @ w_vec + intercept
    # Clip to [0,1] for comparability
    surrogate_preds = np.clip(surrogate_preds, 0, 1)

    corr = np.corrcoef(bb_preds, surrogate_preds)[0, 1]
    mae  = np.mean(np.abs(bb_preds - surrogate_preds))
    return corr, mae


print("Computing LIME fidelity for the true positive sample…")
corr_tp, mae_tp = compute_lime_fidelity(
    lime_explainer, X_test[tp_idx], rf_model.predict_proba,
    num_features=15, num_samples=1000
)
print(f"  Pearson r (surrogate vs black-box): {corr_tp:.4f}")
print(f"  MAE      (surrogate vs black-box): {mae_tp:.4f}")

print("\nComputing LIME fidelity for the false negative sample…")
corr_fn, mae_fn = compute_lime_fidelity(
    lime_explainer, X_test[fn_idx], rf_model.predict_proba,
    num_features=15, num_samples=1000
)
print(f"  Pearson r (surrogate vs black-box): {corr_fn:.4f}")
print(f"  MAE      (surrogate vs black-box): {mae_fn:.4f}")


**Q10 (Fidelity and complexity):**

1. What does it mean if fidelity is low? Can you trust the LIME explanation in that case?
2. How does fidelity change as you increase `num_features`? Why might adding more features not always help?
3. The lecture mentioned that LIME assumes the model is *locally linear*. If the Random Forest is highly non-linear even in a small neighborhood, what do you expect to happen to fidelity?

> *Your answer here*


### 4.7 Stability: LIME explanations under repeated sampling

Because LIME uses random perturbations, running it twice on the same protein gives slightly different explanations. This is the **instability** limitation from the lecture. Here we quantify it.


In [ ]:
# ── 4.7  LIME stability across random seeds ───────────────────────────────────
N_RUNS = 10
TOP_N  = 10    # compare top-10 features across runs

run_weights = {}   # feat_name -> list of weights across runs

for run in range(N_RUNS):
    exp_run = LimeTabularExplainer(
        training_data         = X_train,
        feature_names         = FEATURE_COLS,
        class_names           = ["Non-EV", "EV"],
        mode                  = "classification",
        discretize_continuous = True,
        random_state          = run   # different seed each run
    ).explain_instance(
        data_row     = X_test[tp_idx],
        predict_fn   = rf_model.predict_proba,
        num_features = TOP_N,
        num_samples  = 1000,
    )
    for feat_str, weight in exp_run.as_list(label=1):
        run_weights.setdefault(feat_str, []).append(weight)

# Build summary: features that appear in at least half the runs
stable_feats = {k: v for k, v in run_weights.items() if len(v) >= N_RUNS // 2}
feat_means   = {k: np.mean(v) for k, v in stable_feats.items()}
feat_stds    = {k: np.std(v)  for k, v in stable_feats.items()}
sorted_feats = sorted(feat_means.items(), key=lambda x: abs(x[1]), reverse=True)[:15]

fig, ax = plt.subplots(figsize=(11, 5))
names  = [f[0][:45] for f in sorted_feats]
means  = [f[1] for f in sorted_feats]
stds   = [feat_stds.get(f[0], 0) for f in sorted_feats]
x_pos  = range(len(names))

bars = ax.bar(x_pos, means, yerr=stds, capsize=4,
              color=["#DD8452" if m > 0 else "#4C72B0" for m in means],
              edgecolor="black", alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Mean LIME weight ± std across 10 runs")
ax.set_title(f"LIME stability: protein {tp_idx} — feature weights over {N_RUNS} runs")
ax.axhline(0, color="gray", lw=1, ls="--")
plt.tight_layout()
plt.savefig("lime_stability.png", dpi=120, bbox_inches="tight")
plt.show()

# Compute rank overlap between first and last run
# (a simple way to measure ordering stability)
run0_feats = [f for f, _ in sorted(run_weights.items(),
              key=lambda x: abs(x[1][0]) if x[1] else 0, reverse=True)[:TOP_N]]
run9_feats = [f for f, _ in sorted(run_weights.items(),
              key=lambda x: abs(x[1][-1]) if len(x[1]) > 1 else 0, reverse=True)[:TOP_N]]
overlap = len(set(run0_feats) & set(run9_feats))
print(f"Top-{TOP_N} feature overlap between run 0 and run 9: {overlap}/{TOP_N}")


**Q11.** Looking at the error bars:
- Which features are *stable* across runs (small error bar)?
- Which features are *unstable* (large error bar)?
- What does instability mean for a biologist trying to prioritise features for follow-up experiments?
- How could you reduce instability without changing the model? (Hint: which parameter from Section 4.3?)

> *Your answer here*

---

> ✏️ **STUDENT TASK 4.8:** Run the stability analysis for the MLP instead of the Random Forest. Does the MLP give more or less stable LIME explanations? Why might that be?


In [ ]:
# ── 4.8  STUDENT TASK: LIME stability for MLP ────────────────────────────────
# ========================== STUDENT TASK START ==========================
N_RUNS_MLP = 10
run_weights_mlp = {}

for run in range(N_RUNS_MLP):
    exp_mlp = LimeTabularExplainer(
        training_data         = X_train,
        feature_names         = FEATURE_COLS,
        class_names           = ["Non-EV", "EV"],
        mode                  = "classification",
        discretize_continuous = True,
        random_state          = run
    ).explain_instance(
        data_row     = X_test[tp_idx],
        predict_fn   = mlp_predict_proba,   # <-- MLP
        num_features = 10,
        num_samples  = 1000,
    )
    for feat_str, weight in exp_mlp.as_list(label=1):
        run_weights_mlp.setdefault(feat_str, []).append(weight)

# Summarize stability
stable_mlp  = {k: v for k, v in run_weights_mlp.items() if len(v) >= N_RUNS_MLP // 2}
means_mlp   = {k: np.mean(v) for k, v in stable_mlp.items()}
stds_mlp    = {k: np.std(v)  for k, v in stable_mlp.items()}
sorted_mlp  = sorted(means_mlp.items(), key=lambda x: abs(x[1]), reverse=True)[:10]

fig, ax = plt.subplots(figsize=(11, 4))
names_m = [f[0][:45] for f in sorted_mlp]
means_m = [f[1] for f in sorted_mlp]
stds_m  = [stds_mlp.get(f[0], 0) for f in sorted_mlp]

ax.bar(range(len(names_m)), means_m, yerr=stds_m, capsize=4,
       color=["#DD8452" if m > 0 else "#4C72B0" for m in means_m],
       edgecolor="black", alpha=0.85)
ax.set_xticks(range(len(names_m)))
ax.set_xticklabels(names_m, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Mean LIME weight ± std")
ax.set_title(f"LIME stability on MLP — {N_RUNS_MLP} runs")
ax.axhline(0, color="gray", lw=1, ls="--")
plt.tight_layout()
plt.show()
# ========================== STUDENT TASK END ============================


---
## Section 5: SHAP — Model-Agnostic Methods <a id='shap-ag'></a>

SHAP (SHapley Additive exPlanations) computes **Shapley values** for each feature and prediction. Unlike LIME, SHAP is grounded in game theory: it is the *unique* attribution method satisfying four fairness axioms (efficiency, symmetry, null player, linearity).

In the lecture, we covered three model-agnostic SHAP variants:

| Variant | Strategy | Strength |
|---------|----------|----------|
| **Sampling SHAP** | Sample random coalitions | Simple, unbiased |
| **Permutation SHAP** | Sample random feature orderings | One pass estimates all features |
| **Kernel SHAP** | Weighted regression (like LIME + Shapley kernel) | Lower variance |

All three require a **background dataset** — samples used to marginalize out features that are "absent" from a coalition. The choice and size of the background dataset directly affects the SHAP values. We will explore this carefully.

---

### 5.1 Effect of background sample size

A key practical question: *how many background samples do we need?* More background samples → more accurate marginalization → more reliable SHAP values, but slower computation.

We will use **Kernel SHAP** for this experiment because its results are most sensitive to background sample size.


In [ ]:
# ── 5.1  Effect of background sample size on Kernel SHAP ─────────────────────
# We explain the same true-positive protein (tp_idx) with different background sizes.
# IMPORTANT: this may take a few minutes per run.

background_sizes = [10, 25, 50, 100, 200]
shap_by_bg = {}

# Use a small subset of features to keep runtime manageable
# (full 93-feature Kernel SHAP is slow)
TOP10_IDX  = order[:10]
TOP10_NAMES = [FEATURE_COLS[i] for i in TOP10_IDX]

instance_top10 = X_test[tp_idx:tp_idx+1, :][:, TOP10_IDX]

def rf_predict_top10(X_sub):
    """Insert selected features back into full feature matrix for RF."""
    full = np.tile(X_test[tp_idx], (X_sub.shape[0], 1))
    full[:, TOP10_IDX] = X_sub
    return rf_model.predict_proba(full)[:, 1]

print("Running Kernel SHAP for different background sizes…")
for bg_size in background_sizes:
    rng_bg = np.random.default_rng(SEED)
    bg_idx = rng_bg.choice(len(X_train), size=bg_size, replace=False)
    background_sub = X_train[bg_idx, :][:, TOP10_IDX]

    explainer_k = shap.KernelExplainer(rf_predict_top10, background_sub)
    sv = explainer_k.shap_values(instance_top10, nsamples=256)

    shap_by_bg[bg_size] = sv[0]
    print(f"  bg_size={bg_size:4d}  SHAP values: {sv[0]}")

print("Done.")


In [ ]:
# ── 5.1b  Plot: how SHAP values change with background size ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: heatmap of SHAP values per background size
shap_matrix = np.array([shap_by_bg[s] for s in background_sizes])
sns.heatmap(
    shap_matrix,
    annot=True, fmt=".3f", cmap="coolwarm", center=0,
    xticklabels=[f[:18] for f in TOP10_NAMES],
    yticklabels=[str(s) for s in background_sizes],
    linewidths=0.4,
    ax=axes[0]
)
axes[0].set_xlabel("Feature", fontsize=10)
axes[0].set_ylabel("Background size", fontsize=10)
axes[0].set_title("Kernel SHAP values vs background size", fontsize=11)

# Right: variance of each feature's SHAP value across background sizes
feature_variances = shap_matrix.var(axis=0)
axes[1].barh(
    [f[:30] for f in TOP10_NAMES[::-1]],
    feature_variances[::-1],
    color="#4C72B0", edgecolor="black"
)
axes[1].set_xlabel("Variance of SHAP value across background sizes")
axes[1].set_title("Which features are most sensitive to background choice?")

plt.tight_layout()
plt.savefig("shap_background_effect.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nFeature-wise variance across background sizes:")
for i, feat in enumerate(TOP10_NAMES):
    print(f"  {feat:<35s}  var={feature_variances[i]:.5f}")


**Q12.**
1. Which features show the highest variance in SHAP values across different background sizes? What does that tell you about those features?
2. Why does the background sample matter? (Recall: Kernel SHAP marginalizes out "absent" features by replacing them with background values. What happens if the background is very small and unrepresentative?)
3. The lecture slide "Sanity-checking your explanations" recommended: "Re-run with different background samples — stable rankings ⇒ enough samples." Based on your results, what background size seems sufficient for this dataset?

> *Your answer here*


### 5.2 Sampling SHAP

Sampling SHAP directly estimates the Shapley formula by sampling random coalitions. For each feature, it draws many random subsets, computes marginal contributions, and averages.


In [ ]:
# ── 5.2  Sampling SHAP ───────────────────────────────────────────────────────
# We explain a set of test proteins (subset for speed)
EXPLAIN_IDX = np.random.default_rng(SEED).choice(len(X_test), size=50, replace=False)
X_explain   = X_test[EXPLAIN_IDX]

# Background: 100 training samples (good balance of accuracy vs speed)
bg_idx_100 = np.random.default_rng(SEED).choice(len(X_train), size=100, replace=False)
background_100 = X_train[bg_idx_100]

print("Computing Sampling SHAP (this may take 2–3 minutes)…")
t0 = time.time()

# SamplingExplainer uses the sampling estimator for the Shapley formula
sampling_explainer = shap.SamplingExplainer(
    model      = rf_model.predict_proba,
    data       = background_100,
)
shap_sampling = sampling_explainer.shap_values(X_explain)
# shap_sampling shape: (n_samples, n_features, n_classes) for predict_proba
# We want class=1 (EV)
if isinstance(shap_sampling, list):
    shap_sampling_ev = shap_sampling[1]
else:
    shap_sampling_ev = shap_sampling

t_sampling = time.time() - t0
print(f"Sampling SHAP done in {t_sampling:.1f} s  |  shape: {shap_sampling_ev.shape}")


In [ ]:
# ── 5.2b  Visualize Sampling SHAP ────────────────────────────────────────────
shap_exp_sampling = shap.Explanation(
    values         = shap_sampling_ev,
    base_values    = sampling_explainer.expected_value[1] if hasattr(sampling_explainer.expected_value, '__len__') else sampling_explainer.expected_value,
    data           = X_explain,
    feature_names  = FEATURE_COLS
)

# Global summary: beeswarm plot
plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_exp_sampling, max_display=20, show=False)
plt.title("Sampling SHAP — beeswarm (top 20 features, 50 test proteins)", fontsize=12)
plt.tight_layout()
plt.savefig("shap_sampling_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()

# Local: waterfall for the true positive
tp_local_idx = np.where(EXPLAIN_IDX == tp_idx)[0]
if len(tp_local_idx) > 0:
    shap.plots.waterfall(shap_exp_sampling[tp_local_idx[0]], max_display=15, show=False)
    plt.title(f"Sampling SHAP — waterfall (protein {tp_idx})")
    plt.tight_layout()
    plt.show()


### 5.3 Permutation SHAP

Permutation SHAP estimates Shapley values by averaging marginal contributions over random **orderings** of features. Each permutation contributes one estimate per feature, making it more efficient than sampling SHAP per forward-pass.


In [ ]:
# ── 5.3  Permutation SHAP ────────────────────────────────────────────────────
print("Computing Permutation SHAP…")
t0 = time.time()

perm_explainer = shap.PermutationExplainer(
    model = rf_model.predict_proba,
    masker = shap.maskers.Independent(background_100, max_samples=100)
)

shap_perm = perm_explainer(X_explain)
# shape: (n_samples, n_features, n_classes)
# For binary classification, shap_perm.values has shape (n, f, 2)
shap_perm_ev = shap_perm.values[:, :, 1] if shap_perm.values.ndim == 3 else shap_perm.values

t_perm = time.time() - t0
print(f"Permutation SHAP done in {t_perm:.1f} s  |  shape: {shap_perm_ev.shape}")

shap_exp_perm = shap.Explanation(
    values        = shap_perm_ev,
    base_values   = shap_perm.base_values[:, 1] if shap_perm.base_values.ndim == 2 else shap_perm.base_values,
    data          = X_explain,
    feature_names = FEATURE_COLS
)

plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_exp_perm, max_display=20, show=False)
plt.title("Permutation SHAP — beeswarm (top 20 features, 50 test proteins)", fontsize=12)
plt.tight_layout()
plt.savefig("shap_perm_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()


### 5.4 Kernel SHAP

Kernel SHAP reformulates Shapley value computation as a **weighted linear regression**. It is mathematically equivalent to the Sampling approach but with lower variance because it uses the optimal weighting kernel.

Recall from the lecture: Kernel SHAP = LIME + the Shapley kernel + no sparsity penalty.


In [ ]:
# ── 5.4  Kernel SHAP ─────────────────────────────────────────────────────────
print("Computing Kernel SHAP…")
t0 = time.time()

kernel_explainer = shap.KernelExplainer(
    model  = rf_model.predict_proba,
    data   = background_100,
    link   = "identity"
)

shap_kernel = kernel_explainer.shap_values(X_explain, nsamples=256)
shap_kernel_ev = shap_kernel[1] if isinstance(shap_kernel, list) else shap_kernel

t_kernel = time.time() - t0
print(f"Kernel SHAP done in {t_kernel:.1f} s  |  shape: {shap_kernel_ev.shape}")

shap_exp_kernel = shap.Explanation(
    values        = shap_kernel_ev,
    base_values   = kernel_explainer.expected_value[1] if hasattr(kernel_explainer.expected_value, '__len__') else kernel_explainer.expected_value,
    data          = X_explain,
    feature_names = FEATURE_COLS
)

plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_exp_kernel, max_display=20, show=False)
plt.title("Kernel SHAP — beeswarm (top 20 features, 50 test proteins)", fontsize=12)
plt.tight_layout()
plt.savefig("shap_kernel_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()


### 5.5 Comparing the three model-agnostic SHAP variants


In [ ]:
# ── 5.5  Comparing mean |SHAP| across the three variants ─────────────────────
mean_sampling = np.abs(shap_sampling_ev).mean(axis=0)
mean_perm     = np.abs(shap_perm_ev).mean(axis=0)
mean_kernel   = np.abs(shap_kernel_ev).mean(axis=0)

# Top 20 by kernel SHAP
top20_k = np.argsort(mean_kernel)[::-1][:20]

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(20)
w = 0.25

ax.bar(x - w, mean_sampling[top20_k], w, label="Sampling SHAP", color="#4C72B0", edgecolor="black")
ax.bar(x,     mean_perm[top20_k],     w, label="Permutation SHAP", color="#55A868", edgecolor="black")
ax.bar(x + w, mean_kernel[top20_k],   w, label="Kernel SHAP",    color="#DD8452", edgecolor="black")

ax.set_xticks(x)
ax.set_xticklabels([FEATURE_COLS[i][:20] for i in top20_k], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Mean |SHAP value|")
ax.set_title("Mean absolute SHAP values — three model-agnostic variants (top 20 by Kernel SHAP)")
ax.legend()
plt.tight_layout()
plt.savefig("shap_comparison_agnostic.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nSpearman rank correlation between variant rankings:")
from scipy.stats import spearmanr
r_sk, _ = spearmanr(mean_sampling, mean_kernel)
r_pk, _ = spearmanr(mean_perm,     mean_kernel)
r_sp, _ = spearmanr(mean_sampling, mean_perm)
print(f"  Sampling vs Kernel     : r = {r_sk:.4f}")
print(f"  Permutation vs Kernel  : r = {r_pk:.4f}")
print(f"  Sampling vs Permutation: r = {r_sp:.4f}")

print(f"\nRuntime comparison:")
print(f"  Sampling SHAP   : {t_sampling:.1f} s")
print(f"  Permutation SHAP: {t_perm:.1f} s")
print(f"  Kernel SHAP     : {t_kernel:.1f} s")


**Q13.**
1. Do all three model-agnostic SHAP variants agree on the top features? Which features are consistently important?
2. Do any features show large *disagreements* between methods? What might explain that?
3. Looking at the Spearman correlations, which two methods are most similar to each other? Is that what you expected from the theory?

> *Your answer here*

---

> ✏️ **STUDENT TASK 5.6:** Re-run the Kernel SHAP computation with a background size of **10** instead of 100. How much do the top-feature rankings change? Plot the comparison. What does this tell you about the minimum background size for reliable results?


In [ ]:
# ── 5.6  STUDENT TASK: Kernel SHAP with small background ─────────────────────
# ========================== STUDENT TASK START ==========================
# 1. Create a background set of size 10
# 2. Create a KernelExplainer with this small background
# 3. Compute SHAP values for the same X_explain
# 4. Compare the feature ranking to the Kernel SHAP with 100 background samples
# 5. Compute and print the Spearman rank correlation between the two rankings

bg_idx_10   = np.random.default_rng(SEED).choice(len(X_train), size=10, replace=False)
background_10 = X_train[bg_idx_10]

kernel_explainer_10 = shap.KernelExplainer(
    model  = rf_model.predict_proba,
    data   = background_10,
    link   = "identity"
)
shap_kernel_10 = kernel_explainer_10.shap_values(X_explain, nsamples=256)
shap_kernel_10_ev = shap_kernel_10[1] if isinstance(shap_kernel_10, list) else shap_kernel_10
mean_kernel_10 = np.abs(shap_kernel_10_ev).mean(axis=0)

# Rank correlation
r_10_100, _ = spearmanr(mean_kernel_10, mean_kernel)
print(f"Spearman r (bg=10 vs bg=100): {r_10_100:.4f}")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top20_10 = np.argsort(mean_kernel_10)[::-1][:20]
axes[0].barh([FEATURE_COLS[i][:30] for i in top20_10[::-1]],
             mean_kernel_10[top20_10[::-1]], color="#4C72B0", edgecolor="black")
axes[0].set_title("Kernel SHAP  (bg=10)")

top20_100 = np.argsort(mean_kernel)[::-1][:20]
axes[1].barh([FEATURE_COLS[i][:30] for i in top20_100[::-1]],
             mean_kernel[top20_100[::-1]], color="#DD8452", edgecolor="black")
axes[1].set_title("Kernel SHAP  (bg=100)")

plt.suptitle("Effect of background size on Kernel SHAP feature ranking", fontsize=12)
plt.tight_layout()
plt.show()
# ========================== STUDENT TASK END ============================


---
## Section 6: SHAP — Model-Specific Methods <a id='shap-sp'></a>

Model-agnostic SHAP variants (Kernel, Sampling, Permutation) treat the model as a black box. When we *know* the model architecture, we can exploit it for faster and more exact Shapley values:

| Method | Model | Speed | Exact? |
|--------|-------|-------|--------|
| **TreeSHAP** | Tree ensembles (RF, XGBoost, LightGBM) | Very fast (polynomial time) | Yes |
| **DeepSHAP** | Neural networks (PyTorch, TensorFlow) | Fast (gradient-based) | Approx. |

In the lecture, we highlighted two key advantages of model-specific SHAP:
1. **Speed**: TreeSHAP is orders of magnitude faster than Kernel SHAP for large forests
2. **Reliability**: TreeSHAP is exact; it does not depend on background sample size

---

### 6.1 TreeSHAP for Random Forest

TreeSHAP exploits the tree structure: by walking the decision paths, it can compute exact Shapley values for every sample *without* any sampling approximation.


In [ ]:
# ── 6.1  TreeSHAP for Random Forest ──────────────────────────────────────────
print("Computing TreeSHAP…")
t0_tree = time.time()

tree_explainer = shap.TreeExplainer(
    model           = rf_model,
    feature_perturbation = "tree_path_dependent"  # exact, no background needed
)

# Explain the full test set
shap_tree = tree_explainer.shap_values(X_test)
# For RandomForestClassifier, shap_values returns a list [class0, class1]
shap_tree_ev = shap_tree[1] if isinstance(shap_tree, list) else shap_tree

t_tree = time.time() - t0_tree
print(f"TreeSHAP done in {t_tree:.2f} s  |  shape: {shap_tree_ev.shape}")
print(f"  (Computed for {len(X_test)} proteins with {len(FEATURE_COLS)} features each)")


In [ ]:
# ── 6.1b  TreeSHAP: global and local visualizations ──────────────────────────
shap_exp_tree = shap.Explanation(
    values        = shap_tree_ev,
    base_values   = tree_explainer.expected_value[1] if hasattr(tree_explainer.expected_value, '__len__') else tree_explainer.expected_value,
    data          = X_test,
    feature_names = FEATURE_COLS
)

# Beeswarm (global)
plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_exp_tree, max_display=20, show=False)
plt.title("TreeSHAP — beeswarm (full test set)", fontsize=12)
plt.tight_layout()
plt.savefig("treeshap_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()

# Waterfall for one protein (true positive)
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_exp_tree[tp_idx], max_display=15, show=False)
plt.title(f"TreeSHAP — waterfall (protein {tp_idx}, true positive)")
plt.tight_layout()
plt.savefig("treeshap_waterfall_tp.png", dpi=120, bbox_inches="tight")
plt.show()

# Bar plot (mean |SHAP|)
plt.figure(figsize=(9, 6))
shap.plots.bar(shap_exp_tree, max_display=20, show=False)
plt.title("TreeSHAP — mean |SHAP| (global importance)", fontsize=12)
plt.tight_layout()
plt.savefig("treeshap_bar.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── 6.1c  TreeSHAP: dependence plot ──────────────────────────────────────────
# A SHAP dependence plot shows ϕᵢ vs xᵢ across all test proteins.
# The color represents a second feature (automatically chosen as the most
# interacting feature), revealing pairwise interactions.
top1_feat = FEATURE_COLS[np.argsort(np.abs(shap_tree_ev).mean(axis=0))[::-1][0]]

plt.figure(figsize=(8, 5))
shap.dependence_plot(
    top1_feat, shap_tree_ev, X_test,
    feature_names=FEATURE_COLS,
    show=False
)
plt.title(f"TreeSHAP dependence plot — {top1_feat}", fontsize=12)
plt.tight_layout()
plt.savefig("treeshap_dependence.png", dpi=120, bbox_inches="tight")
plt.show()


**Q14.** Interpret the TreeSHAP beeswarm plot:
- Which features push toward EV (positive SHAP)?
- Which push away from EV (negative SHAP)?
- Look at the color coding: for the top feature, do high values (red) increase or decrease the EV probability? Does this match your biological intuition?

**Q15.** Look at the dependence plot:
- What does the shape of the scatter tell you about how the top feature affects predictions?
- The color of each dot represents a second, automatically-selected feature. Where do you see *clustering by color*? That indicates an **interaction** between the two features. Can you interpret this biologically?

> *Your answer here (Q14 + Q15)*


### 6.2 DeepSHAP for the MLP

DeepSHAP combines the Shapley framework with backpropagation (via DeepLIFT). Instead of sampling coalitions, it uses gradients to propagate attributions through the network layers.

**Key requirement:** DeepSHAP needs a set of **background samples** as a reference point. It computes the difference `f(x) - f(background)` and attributes it layer-by-layer.


In [ ]:
# ── 6.2  DeepSHAP for the MLP ────────────────────────────────────────────────
# DeepSHAP works with PyTorch tensors.
# We pass the scaled data (as the MLP was trained on scaled inputs).

print("Computing DeepSHAP…")
t0_deep = time.time()

# Background: 100 training samples (scaled)
bg_scaled = torch.tensor(X_train_scaled[bg_idx_100], dtype=torch.float32).to(DEVICE)

# Wrap the MLP so it outputs a single value (probability for EV class)
class MLPForShap(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        logits = self.model(x)
        return torch.sigmoid(logits).unsqueeze(-1)   # shape: (B, 1)


mlp_shap_wrapper = MLPForShap(mlp).to(DEVICE)
mlp_shap_wrapper.eval()

# Samples to explain (first 50 test proteins, scaled)
X_explain_scaled = torch.tensor(X_test_scaled[:50], dtype=torch.float32).to(DEVICE)

deep_explainer = shap.DeepExplainer(mlp_shap_wrapper, bg_scaled)
shap_deep = deep_explainer.shap_values(X_explain_scaled)

# shap_deep is a list with one element (single output), each shape (N, features)
shap_deep_ev = shap_deep[0] if isinstance(shap_deep, list) else shap_deep

t_deep = time.time() - t0_deep
print(f"DeepSHAP done in {t_deep:.2f} s  |  shape: {np.array(shap_deep_ev).shape}")


In [ ]:
# ── 6.2b  DeepSHAP visualizations ────────────────────────────────────────────
shap_deep_np = np.array(shap_deep_ev)

shap_exp_deep = shap.Explanation(
    values        = shap_deep_np,
    base_values   = deep_explainer.expected_value[0] if hasattr(deep_explainer.expected_value, '__len__') else deep_explainer.expected_value,
    data          = X_test_scaled[:50],
    feature_names = FEATURE_COLS
)

# Beeswarm
plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_exp_deep, max_display=20, show=False)
plt.title("DeepSHAP — beeswarm (50 test proteins, MLP)", fontsize=12)
plt.tight_layout()
plt.savefig("deepshap_beeswarm.png", dpi=120, bbox_inches="tight")
plt.show()

# Waterfall for true positive
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_exp_deep[tp_idx], max_display=15, show=False)
plt.title(f"DeepSHAP — waterfall (protein {tp_idx}, MLP)")
plt.tight_layout()
plt.savefig("deepshap_waterfall.png", dpi=120, bbox_inches="tight")
plt.show()


### 6.3 Speed comparison: TreeSHAP vs Kernel SHAP

One of the main selling points of TreeSHAP is speed. We already have the runtimes from above. Let's make the comparison explicit and also scale it to the full test set.


In [ ]:
# ── 6.3  Speed comparison ─────────────────────────────────────────────────────
# We already have t_tree, t_kernel, t_deep, t_sampling, t_perm from above.
# Compute per-sample speed.

n_explain = len(X_explain)  # 50 proteins

speed_data = {
    "Sampling SHAP\n(50 samples)": t_sampling,
    "Permutation SHAP\n(50 samples)": t_perm,
    "Kernel SHAP\n(50 samples)": t_kernel,
    "TreeSHAP\n(full test set)": t_tree,
    "DeepSHAP\n(50 samples)": t_deep,
}

fig, ax = plt.subplots(figsize=(10, 5))
colors_speed = ["#4C72B0", "#4C72B0", "#4C72B0", "#55A868", "#DD8452"]
bars = ax.bar(speed_data.keys(), speed_data.values(),
              color=colors_speed, edgecolor="black")

for bar, val in zip(bars, speed_data.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}s", ha="center", fontsize=10)

ax.set_ylabel("Wall-clock time (seconds)")
ax.set_title("Runtime comparison: model-agnostic vs model-specific SHAP", fontsize=12)

# Color legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#4C72B0", label="Model-agnostic (black-box)"),
    Patch(facecolor="#55A868", label="TreeSHAP (RF — exact)"),
    Patch(facecolor="#DD8452", label="DeepSHAP (MLP — gradient)"),
]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.savefig("shap_speed_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"TreeSHAP computed {len(X_test)} proteins in {t_tree:.2f}s")
print(f"Kernel SHAP computed {n_explain} proteins in {t_kernel:.2f}s")
print(f"Projected Kernel SHAP for {len(X_test)} proteins: "
      f"{t_kernel * len(X_test) / n_explain:.0f}s ≈ "
      f"{t_kernel * len(X_test) / n_explain / 60:.1f} min")


**Q16.**
1. By roughly how many times is TreeSHAP faster than Kernel SHAP for the same number of proteins?
2. The lecture stated that TreeSHAP is **exact** while Kernel SHAP is **approximate**. What does "exact" mean here? (Hint: what property of Shapley values is exactly satisfied?)
3. DeepSHAP is also an approximation — why? (Recall: it uses the *baseline* value function rather than the marginal value function, and approximates at each non-linearity.)

> *Your answer here*


### 6.4 Reliability: TreeSHAP vs Kernel SHAP

Beyond speed, TreeSHAP should give more *reliable* (less variable) results because it is exact — it does not depend on how many background samples you use or how many coalitions you sample.

Let's verify this by comparing the feature rankings from TreeSHAP and Kernel SHAP on the same proteins.


In [ ]:
# ── 6.4  Reliability: TreeSHAP vs Kernel SHAP ────────────────────────────────
# Compare mean |SHAP| rankings on the 50 proteins we explained with both methods.
shap_tree_sub = shap_tree_ev[EXPLAIN_IDX]   # TreeSHAP for the same 50 proteins

mean_tree_sub   = np.abs(shap_tree_sub).mean(axis=0)
mean_kernel_sub = np.abs(shap_kernel_ev).mean(axis=0)

# Spearman rank correlation
r_tk, _ = spearmanr(mean_tree_sub, mean_kernel_sub)
print(f"Spearman rank correlation (TreeSHAP vs Kernel SHAP): r = {r_tk:.4f}")

# Plot comparison
top30_tree   = np.argsort(mean_tree_sub)[::-1][:30]
top30_kernel = np.argsort(mean_kernel_sub)[::-1][:30]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh([FEATURE_COLS[i][:30] for i in top30_tree[::-1]],
             mean_tree_sub[top30_tree[::-1]], color="#55A868", edgecolor="black")
axes[0].set_title(f"TreeSHAP — mean |ϕ|\n(exact, 50 proteins)", fontsize=11)
axes[0].set_xlabel("Mean |SHAP value|")

axes[1].barh([FEATURE_COLS[i][:30] for i in top30_kernel[::-1]],
             mean_kernel_sub[top30_kernel[::-1]], color="#DD8452", edgecolor="black")
axes[1].set_title(f"Kernel SHAP — mean |ϕ|\n(approximate, bg=100, 50 proteins)", fontsize=11)
axes[1].set_xlabel("Mean |SHAP value|")

plt.suptitle(f"Feature importance ranking: TreeSHAP vs Kernel SHAP  (Spearman r = {r_tk:.3f})",
             fontsize=12)
plt.tight_layout()
plt.savefig("treeshap_vs_kernel.png", dpi=120, bbox_inches="tight")
plt.show()


**Q17.**
1. Do TreeSHAP and Kernel SHAP agree on the top features? What is the Spearman correlation?
2. For which features do they *disagree* most? Can you think of a reason — are those features correlated with others?
3. In a real biological study, would you prefer TreeSHAP or Kernel SHAP for a Random Forest model? Justify your answer.

> *Your answer here*

---

> ✏️ **STUDENT TASK 6.5:** Compute TreeSHAP SHAP interaction values for the top 10 features using `shap.TreeExplainer` with `model_output='raw'` and `feature_perturbation='tree_path_dependent'`. Then call `tree_explainer.shap_interaction_values(X_test[:100])` and visualize the interaction matrix as a heatmap. Which feature pair shows the strongest interaction? Can you interpret it biologically?


In [ ]:
# ── 6.5  STUDENT TASK: SHAP interaction values ───────────────────────────────
# ========================== STUDENT TASK START ==========================
# Compute and visualize SHAP interaction values.
# Note: this requires a non-probability output; use raw tree output.

tree_explainer_raw = shap.TreeExplainer(
    model                  = rf_model,
    feature_perturbation   = "tree_path_dependent"
)

# Interaction values: shape (N, features, features)
# This may take 2–3 minutes
print("Computing SHAP interaction values for first 100 test proteins…")
shap_interact = tree_explainer_raw.shap_interaction_values(X_test[:100])
# For RandomForest it's a list [class0, class1]
if isinstance(shap_interact, list):
    shap_interact_ev = shap_interact[1]
else:
    shap_interact_ev = shap_interact

# Mean absolute interaction matrix
mean_interact = np.abs(shap_interact_ev).mean(axis=0)

# Focus on top 15 features
top15 = np.argsort(np.abs(shap_tree_ev[:100]).mean(axis=0))[::-1][:15]
interact_sub = mean_interact[np.ix_(top15, top15)]
feat_labels  = [FEATURE_COLS[i][:20] for i in top15]

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(interact_sub, xticklabels=feat_labels, yticklabels=feat_labels,
            cmap="YlOrRd", annot=True, fmt=".3f", linewidths=0.3, ax=ax)
ax.set_title("Mean absolute SHAP interaction values (top 15 features)", fontsize=12)
plt.tight_layout()
plt.savefig("shap_interactions.png", dpi=120, bbox_inches="tight")
plt.show()

# Find the strongest off-diagonal interaction
np.fill_diagonal(interact_sub, 0)
i_max, j_max = np.unravel_index(np.argmax(interact_sub), interact_sub.shape)
print(f"Strongest interaction: {feat_labels[i_max]} <-> {feat_labels[j_max]}  "
      f"(mean |interaction| = {interact_sub[i_max, j_max]:.4f})")
# ========================== STUDENT TASK END ============================


---
## Section 7: Synthesis — Comparing Methods and Biological Interpretation <a id='synthesis'></a>

Now that you have used PDP, ICE, ALE, LIME, Sampling SHAP, Permutation SHAP, Kernel SHAP, TreeSHAP, and DeepSHAP — it's time to step back and compare.

### 7.1 Feature importance consensus across methods


In [ ]:
# ── 7.1  Feature importance consensus ────────────────────────────────────────
# Collect top-10 feature rankings from each method.

# Permutation importance
top10_perm_imp = [FEATURE_COLS[i] for i in order[:10]]

# LIME (mean |weight| across 10 stable runs)
lime_feat_importance = {k: abs(np.mean(v)) for k, v in run_weights.items() if len(v) >= 5}
top10_lime = sorted(lime_feat_importance, key=lime_feat_importance.get, reverse=True)[:10]
# Strip LIME discretisation tags to match feature names
top10_lime_clean = []
for lime_f in top10_lime:
    matched = [f for f in FEATURE_COLS if f in lime_f]
    if matched: top10_lime_clean.append(matched[0])
top10_lime_clean = list(dict.fromkeys(top10_lime_clean))[:10]  # deduplicate

# Kernel SHAP
top10_kernel_shap = [FEATURE_COLS[i] for i in np.argsort(mean_kernel)[::-1][:10]]

# TreeSHAP
mean_tree_all = np.abs(shap_tree_ev).mean(axis=0)
top10_tree_shap = [FEATURE_COLS[i] for i in np.argsort(mean_tree_all)[::-1][:10]]

# Build consensus table
all_methods = {
    "Permutation Importance": top10_perm_imp,
    "LIME (stable avg)":      top10_lime_clean[:10],
    "Kernel SHAP":            top10_kernel_shap,
    "TreeSHAP":               top10_tree_shap,
}

print(f"{'Rank':<5}", end="")
for method in all_methods:
    print(f"  {method:<30}", end="")
print()
print("-" * (5 + 32 * len(all_methods)))

for rank in range(1, 11):
    print(f"{rank:<5}", end="")
    for method, feat_list in all_methods.items():
        feat = feat_list[rank-1] if rank <= len(feat_list) else "-"
        print(f"  {feat:<30}", end="")
    print()

# Which features appear in top-10 of ALL methods?
all_top10_sets = [set(v[:10]) for v in all_methods.values()]
consensus = all_top10_sets[0].intersection(*all_top10_sets[1:])
print(f"\nFeatures in top-10 of ALL methods: {consensus if consensus else '(none)'}")

union_top10 = all_top10_sets[0].union(*all_top10_sets[1:])
majority = {f for f in union_top10
            if sum(f in s for s in all_top10_sets) >= len(all_methods) // 2 + 1}
print(f"Features in top-10 of majority (≥{len(all_methods)//2+1}/{len(all_methods)}) of methods: {majority}")


**Q18.** Compare the feature rankings across methods:
1. Which features appear in the top-10 of **all** methods? These are the most robust findings.
2. Which features appear in only one method's top-10? Should you trust a feature that only one method identifies as important?
3. Does the consensus list match your biological hypotheses from DAY1-Q3? If not, what did you get wrong — and why?

> *Your answer here*


### 7.2 LIME vs SHAP: a head-to-head comparison

Recall from the lecture:

> **Bottom line: Kernel SHAP = LIME with the unique kernel/loss combination that recovers the Shapley value.**

This means LIME and Kernel SHAP are theoretically related, but differ in their weighting kernel, loss function, and whether they enforce sparsity. Let's test this empirically.


In [ ]:
# ── 7.2  LIME vs SHAP for a single protein ────────────────────────────────────
# ========================== STUDENT TASK START ==========================
# STUDENT TASK 7.2: Compare LIME and Kernel SHAP attributions side by side
# for the true-positive protein (tp_idx).
#
# 1. Get LIME weights (use the standard lime_explainer, num_features=15)
# 2. Get Kernel SHAP values for that protein (already computed in shap_kernel_ev)
# 3. For each feature in the LIME top-15, find the corresponding Kernel SHAP value
# 4. Plot a scatter: LIME weight (x-axis) vs SHAP value (y-axis)
# 5. Compute the Pearson/Spearman correlation
# 6. Which features agree? Which disagree the most?

exp_vs = lime_explainer.explain_instance(
    data_row     = X_test[tp_idx],
    predict_fn   = rf_model.predict_proba,
    num_features = 20,
    num_samples  = 2000,
)
lime_vals = dict(exp_vs.as_list(label=1))

# Map LIME feature strings to feature indices
lime_feat_vals, shap_feat_vals, feat_labels_comp = [], [], []
for feat_str, lime_w in lime_vals.items():
    for fi, fname in enumerate(FEATURE_COLS):
        if fname in feat_str:
            lime_feat_vals.append(lime_w)
            shap_feat_vals.append(shap_kernel_ev[np.where(EXPLAIN_IDX == tp_idx)[0][0], fi]
                                  if tp_idx in EXPLAIN_IDX else 0.0)
            feat_labels_comp.append(fname[:25])
            break

from scipy.stats import pearsonr, spearmanr as sp2
r_pearson,   _ = pearsonr(lime_feat_vals, shap_feat_vals)
r_spearman, _  = sp2(lime_feat_vals, shap_feat_vals)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(lime_feat_vals, shap_feat_vals, color="#4C72B0", s=60, edgecolors="black", alpha=0.8)
for lv, sv, lab in zip(lime_feat_vals, shap_feat_vals, feat_labels_comp):
    ax.annotate(lab, (lv, sv), fontsize=7, alpha=0.7)
ax.axhline(0, color="gray", lw=1, ls="--")
ax.axvline(0, color="gray", lw=1, ls="--")
ax.set_xlabel("LIME weight")
ax.set_ylabel("Kernel SHAP value")
ax.set_title(f"LIME vs Kernel SHAP — protein {tp_idx}
"
             f"Pearson r={r_pearson:.3f}  Spearman r={r_spearman:.3f}")
plt.tight_layout()
plt.savefig("lime_vs_kernelshap.png", dpi=120, bbox_inches="tight")
plt.show()
# ========================== STUDENT TASK END ============================


**Q19.**
1. Do LIME and Kernel SHAP agree on which features are most important for this protein?
2. Where do they disagree? Look at specific features where the sign (direction) differs. Is that possible, or is it a sign of instability?
3. LIME enforces sparsity (it only uses `num_features` features) while Kernel SHAP assigns a value to every feature. How does that affect your comparison?
4. Given the theoretical relationship between LIME and Kernel SHAP, when would you choose one over the other in practice?

> *Your answer here*


### 7.3 Biological interpretation

We have been computing feature attributions all assignment. Now let's connect the results back to the biology.

EV protein sorting is known to be influenced by several mechanisms:
- **Ubiquitination and PTMs**: proteins destined for multivesicular bodies (MVBs) are often ubiquitinated
- **Transmembrane domains**: membrane proteins can be sorted into EVs via membrane topology
- **Hydrophobicity and surface properties**: hydrophobic patches can drive membrane association
- **Phosphorylation**: phosphorylation can create sorting signals
- **Coiled-coil domains**: some EV-sorted proteins have coiled-coil interaction domains

---

> ✏️ **STUDENT TASK 7.3 (Written reflection — no code required)**
>
> Based on all the XAI methods you applied in this assignment, write a short (1–2 paragraph) biological interpretation of your results.
>
> **Your answer should include:**
> 1. The top 3–5 features consistently identified as important by multiple methods
> 2. Whether these features align with known EV biology (cite the mechanisms above or others you know)
> 3. One feature that was surprisingly important or unimportant
> 4. A caution: which findings might be artefacts of feature correlation rather than true biological signal?
>
> Remember the lecture warning: **feature attribution ≠ biological causation**. Your answer should reflect that nuance.

> *Your answer here*


### 7.4 Method selection guide

As a practical summary, we revisit the method selection question from the lecture.

> ✏️ **STUDENT TASK 7.4:** Fill in the table below based on what you observed in this assignment.
>
> For each method, summarize in your own words the main strength, main weakness, and a scenario in biology where you would recommend using it.

| Method | Main strength | Main weakness | Best use case in biology |
|--------|--------------|---------------|--------------------------|
| Permutation importance | | | |
| PDP | | | |
| ICE | | | |
| ALE | | | |
| LIME | | | |
| Kernel SHAP | | | |
| TreeSHAP | | | |
| DeepSHAP | | | |

> *Fill in the table above*


In [ ]:
# ── 7.5  Final summary figure ─────────────────────────────────────────────────
# A single figure comparing mean feature importance from four key methods side by side.

methods_final = {
    "Permutation\nImportance":  (perm_mean,          FEATURE_COLS),
    "Kernel SHAP\n(approx.)":   (mean_kernel,         FEATURE_COLS),
    "TreeSHAP\n(exact)":        (mean_tree_all,       FEATURE_COLS),
    "DeepSHAP\n(MLP)":          (np.abs(shap_deep_np).mean(axis=0), FEATURE_COLS),
}

# Use top-15 by TreeSHAP as reference
top15_ref = np.argsort(mean_tree_all)[::-1][:15]
feat_labels_final = [FEATURE_COLS[i][:22] for i in top15_ref]

fig, axes = plt.subplots(1, 4, figsize=(20, 7), sharey=True)
palette = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

for ax, (method_name, (importance, feat_list)), color in zip(axes, methods_final.items(), palette):
    # Normalize to [0,1] for comparability
    vals = importance[top15_ref]
    vals_norm = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
    ax.barh(feat_labels_final[::-1], vals_norm[::-1], color=color, edgecolor="black", alpha=0.85)
    ax.set_title(method_name, fontsize=11)
    ax.set_xlabel("Normalized importance", fontsize=9)
    ax.axvline(0.5, color="gray", lw=0.5, ls="--")

plt.suptitle("Feature importance consensus — four methods (normalized, top 15 by TreeSHAP)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("final_feature_importance_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


---
## Wrap-up

Congratulations — you have applied the full XAI pipeline from the lecture to a real protein biology dataset.

### Key takeaways

1. **Global vs local:** PDP, ALE, and permutation importance describe the model overall. LIME and SHAP explain individual predictions — and those per-protein explanations can differ widely even when global importances look similar.

2. **LIME is intuitive but fragile:** Its results depend on the kernel width, number of perturbations, and random seed. Use large `num_samples`, average across runs, and always check fidelity.

3. **SHAP is principled:** Shapley values satisfy four fairness axioms, making them more theoretically grounded than LIME. But they also depend on the choice of background samples and value function (marginal vs conditional).

4. **TreeSHAP is the practical default for tree models:** It is exact, fast, and does not require background sample tuning. If you are using a Random Forest or XGBoost in your research, always prefer TreeSHAP.

5. **Feature attribution ≠ causation:** A high SHAP value for phosphorylation does not mean phosphorylation *causes* EV sorting — it means the model uses that feature. Biological validation (knockouts, mutants, perturbations) is always necessary.

---

### Exam preparation checklist

Before the oral exam, make sure you can:

- [ ] Explain what PDP, ICE, and ALE each measure and when ALE is preferred over PDP
- [ ] Walk through the LIME pipeline step by step (perturbation, weighting, surrogate, coefficients)
- [ ] Explain the four Shapley axioms in plain English with an example
- [ ] Explain the difference between Sampling SHAP, Permutation SHAP, and Kernel SHAP
- [ ] Explain when to use TreeSHAP vs DeepSHAP vs Kernel SHAP
- [ ] Give one example of how correlated features can mislead LIME or SHAP
- [ ] Describe what "fidelity" means for LIME and why it matters

---

*End of Assignment 1*
